In [ ]:
"""
================================================================================
REGIONAL GCM ENSEMBLE TEMPERATURE + PRECIPITATION ANOMALIES
   Tropical / Arid / Temperate Andes 

  - 9-GCM CMIP6 ensemble: BCC-CSM2-MR, CAMS-CSM1-0, CESM2, EC-Earth3-Veg,
    FGOALS-f3-L, GFDL-ESM4, MPI-ESM1-2-HR, MRI-ESM2-0, NorESM2-MM
  - 4 SSPs: 1-2.6 / 2-4.5 / 3-7.0 / 5-8.5

  - Anomalies:
      Temperature:    T(t) - T_baseline       [degC]
      Precipitation: (P(t) - P_baseline) / P_baseline * 100  [%]
  - TWO regional aggregation methods (one figure each):
      AREA-WEIGHTED:  weight each glacier's climate by its RGI area

      EQUAL-WEIGHTED: average across glaciers with equal weight
    
  - Time series: ensemble mean +/- 1 sd across the 9 GCMs, annual
  - 2071-2100 climatology annotated on each panel

================================================================================
HOW TO USE
================================================================================
Step 1: Set CATEGORY below to 'Tropical', 'Arid', or 'Temperate'.
Step 2: Make sure the matching CSV is next to this notebook:
          tropical_rgi_ids.csv / arid_rgi_ids.csv / temperate_rgi_ids.csv
        Same files used by your Phase 2 regional script.
Step 3: Run.
Step 4: For the next region, change CATEGORY and re-run.

================================================================================
"""

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import oggm.cfg as cfg
from oggm import utils, workflow, DEFAULT_BASE_URL
from oggm.shop import gcm_climate

# ===========================================================================
# CATEGORY REGISTRY - edit CSV paths if your files live elsewhere
# ===========================================================================
CATEGORIES = {
    'Tropical':  {
        'label':    'Tropical Andes',
        'csv_path': 'tropical_rgi_ids.csv',
    },
    'Arid':      {
        'label':    'Arid Andes',
        'csv_path': 'arid_rgi_ids.csv',
    },
    'Temperate': {
        'label':    'Temperate Andes',
        'csv_path': 'temperate_rgi_ids.csv',
    },
}

# === CHANGE THIS LINE TO SWITCH CATEGORIES =================================
CATEGORY = 'Tropical'   # must match a key in CATEGORIES above
# ===========================================================================

# ---------------------------------------------------------------------------
# OGGM initialisation 
# ---------------------------------------------------------------------------
cfg.initialize(logging_level='WARNING')
cfg.PARAMS['continue_on_error']        = True
cfg.PARAMS['min_ice_thick_for_length'] = 1
cfg.PARAMS['store_model_geometry']     = True
cfg.PARAMS['use_multiprocessing']      = True   # needed for ~500 glaciers

# Use the SAME working directory pattern as the Phase 2 regional script,
# so cached gcm_data files (if present from earlier runs) are re-used.
cfg.PATHS['working_dir'] = utils.gettempdir(
    f'OGGM_{CATEGORY}_regional', reset=False)
print(f'OGGM working directory: {cfg.PATHS["working_dir"]}')

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------
GCMS = [
    'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CESM2', 'EC-Earth3-Veg',
    'FGOALS-f3-L', 'GFDL-ESM4', 'MPI-ESM1-2-HR',
    'MRI-ESM2-0', 'NorESM2-MM',
]
SSPS = ['ssp126', 'ssp245', 'ssp370', 'ssp585']

SSP_COLOURS = {
    'ssp126': 'green',     'ssp245': 'blue',
    'ssp370': '#ff7f0e',   'ssp585': 'red',
}
SSP_LABELS = {
    'ssp126': 'SSP1-2.6', 'ssp245': 'SSP2-4.5',
    'ssp370': 'SSP3-7.0', 'ssp585': 'SSP5-8.5',
}

# Bremen URL templates (only needed if gcm_data files are missing and
# need re-processing)
BP_SSP = ('https://cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/'
          '{gcm}/{gcm}_{ssp}_r1i1p1f1_pr.nc')
BT_SSP = ('https://cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/'
          '{gcm}/{gcm}_{ssp}_r1i1p1f1_tas.nc')

BASELINE = (1995, 2014)
PROJ_END = (2071, 2100)
PLOT_START = 2020

cat_info  = CATEGORIES[CATEGORY]
label     = cat_info['label']
csv_path  = cat_info['csv_path']

OUTDIR = os.path.join(os.getcwd(), f'{CATEGORY}_climate_anomaly_outputs')
os.makedirs(OUTDIR, exist_ok=True)
print(f'Output folder: {OUTDIR}')

# ---------------------------------------------------------------------------
# Load RGI IDs from the same CSV used by the Phase 2 regional script
# ---------------------------------------------------------------------------
if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"RGI ID file not found: {csv_path}\n"
        f"Please create '{csv_path}' next to this notebook with one\n"
        f"RGI60-... identifier per line.")

df_ids = pd.read_csv(csv_path, header=None)
if str(df_ids.iloc[0, 0]).lower().startswith('rgi') is False:
    df_ids = pd.read_csv(csv_path)
    rgi_ids = df_ids[df_ids.columns[0]].astype(str).str.strip().tolist()
else:
    rgi_ids = df_ids.iloc[:, 0].astype(str).str.strip().tolist()
rgi_ids = [r for r in rgi_ids if r.startswith('RGI60-')]
rgi_ids = list(dict.fromkeys(rgi_ids))   # dedupe
print(f'Loaded {len(rgi_ids)} RGI IDs from {csv_path}')

# ---------------------------------------------------------------------------
# Initialise glacier directories (re-uses Phase 2 cache if present)
# ---------------------------------------------------------------------------
print('\nInitialising glacier directories...')
gdirs = workflow.init_glacier_directories(
    rgi_ids,
    from_prepro_level=5,
    prepro_base_url=DEFAULT_BASE_URL,
)
gdirs = sorted(gdirs, key=lambda g: g.rgi_area_km2, reverse=True)
total_area = sum(g.rgi_area_km2 for g in gdirs)
print(f'  {len(gdirs)} glaciers,  total area {total_area:.2f} km^2')
print(f'  largest: {gdirs[0].rgi_id} ({gdirs[0].rgi_area_km2:.2f} km^2)')
print(f'  smallest: {gdirs[-1].rgi_id} '
      f'({gdirs[-1].rgi_area_km2:.3f} km^2)')

# Pre-compute the two weight vectors
areas        = np.array([g.rgi_area_km2 for g in gdirs])
weights_area = areas / areas.sum()
weights_eq   = np.ones_like(areas) / len(areas)

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def ensure_gcm_data(gdirs, gcm, ssp):
    """
    Check whether all gdirs have the gcm_data file for this (gcm, ssp).
    If not, re-process from Bremen. Returns True on success.
    """
    fid = f'_CMIP6_{gcm}_{ssp}'
    missing = [g for g in gdirs
               if not os.path.exists(g.get_filepath('gcm_data',
                                                    filesuffix=fid))]
    if not missing:
        return True
    print(f'    {len(missing)}/{len(gdirs)} gdirs missing {fid}, '
          f'reprocessing...')
    try:
        ft = utils.file_downloader(BT_SSP.format(gcm=gcm, ssp=ssp))
        fp = utils.file_downloader(BP_SSP.format(gcm=gcm, ssp=ssp))
        if ft is None or fp is None:
            print(f'    SKIP: GCM file not on Bremen server')
            return False
        workflow.execute_entity_task(
            gcm_climate.process_cmip_data, gdirs,
            filesuffix=fid, fpath_temp=ft, fpath_precip=fp,
        )
        return True
    except Exception as e:
        print(f'    SKIP: reprocessing failed: {e}')
        return False

def read_gcm_climate(gdir, gcm, ssp):
    """
    Read the bias-corrected GCM climate for one glacier.
    Returns (temp_annual, prcp_annual) as pandas Series indexed by year,
    or (None, None) if the file is missing.

    Uses netCDF4 directly + numpy bincount instead of xarray + pandas
    groupby. This is ~5-10x faster per call, which matters when called
    591 x 36 = 21,276 times for a 591-glacier region.
    """
    from netCDF4 import Dataset, num2date

    fid = f'_CMIP6_{gcm}_{ssp}'
    path = gdir.get_filepath('gcm_data', filesuffix=fid)
    if not os.path.exists(path):
        return None, None

    with Dataset(path, 'r') as nc:
        temp_m = nc['temp'][:].astype(np.float64)
        prcp_m = nc['prcp'][:].astype(np.float64)
        t_var = nc['time']
        t_units = t_var.units
        t_calendar = getattr(t_var, 'calendar', 'standard')
        t_raw = t_var[:]

    try:
        cf_dates = num2date(t_raw, units=t_units, calendar=t_calendar)
        # cf_dates is array of cftime/datetime objects
        years = np.fromiter((d.year for d in cf_dates),
                            dtype=np.int32, count=len(cf_dates))
    except Exception:
        # Last-resort fallback via xarray (slow but always works)
        ds = xr.open_dataset(path, decode_times=True)
        time_vals = ds.time.values
        ds.close()
        try:
            years = pd.DatetimeIndex(time_vals).year.values.astype(np.int32)
        except (TypeError, ValueError):
            years = np.array([int(t.year) for t in time_vals],
                             dtype=np.int32)

    # Aggregate monthly -> annual using numpy bincount
    # (much faster than pandas groupby for this volume of data)
    yr_min = int(years.min())
    yr_idx = years - yr_min                          # 0-based year index
    n_years = int(years.max() - yr_min + 1)
    counts = np.bincount(yr_idx, minlength=n_years)
    temp_sum = np.bincount(yr_idx, weights=temp_m, minlength=n_years)
    prcp_sum = np.bincount(yr_idx, weights=prcp_m, minlength=n_years)

    # Guard against divide-by-zero for missing years
    with np.errstate(invalid='ignore', divide='ignore'):
        temp_ann_arr = np.where(counts > 0, temp_sum / counts, np.nan)
    prcp_ann_arr = prcp_sum   # annual SUM of monthly mm

    year_index = np.arange(yr_min, yr_min + n_years)
    temp_ann = pd.Series(temp_ann_arr, index=year_index)
    prcp_ann = pd.Series(prcp_ann_arr, index=year_index)
    return temp_ann, prcp_ann

# ---------------------------------------------------------------------------
# Build per-GCM, per-SSP regional time series under BOTH weightings 
# ---------------------------------------------------------------------------
print(f'\nReading bias-corrected GCM climate from gcm_data files...')
print(f'Baseline: {BASELINE[0]}-{BASELINE[1]}')
print(f'Plot range: {PLOT_START}-{PROJ_END[1]}\n')

results = {scheme: {var: {ssp: pd.DataFrame() for ssp in SSPS}
                    for var in ('tas', 'pr')}
           for scheme in ('area', 'equal')}

for gcm in GCMS:
    print(f'>>> {gcm}')
    for ssp in SSPS:
        if not ensure_gcm_data(gdirs, gcm, ssp):
            continue

        
        t_each, p_each = [], []
        for g in gdirs:
            t_ann, p_ann = read_gcm_climate(g, gcm, ssp)
            t_each.append(t_ann)
            p_each.append(p_ann)

        valid_idx = [i for i, t in enumerate(t_each) if t is not None]
        if not valid_idx:
            print(f'    {ssp}: no readable gcm_data files, skipping')
            continue

        
        t_df = pd.concat([t_each[i] for i in valid_idx], axis=1)
        p_df = pd.concat([p_each[i] for i in valid_idx], axis=1)

        
        for scheme, w_full in (('area', weights_area),
                               ('equal', weights_eq)):
            w = w_full[valid_idx]
            w = w / w.sum()
            t_reg = (t_df * w).sum(axis=1)
            p_reg = (p_df * w).sum(axis=1)

            
            t_base_slice = t_reg.loc[
                (t_reg.index >= BASELINE[0])
                & (t_reg.index <= BASELINE[1])]
            p_base_slice = p_reg.loc[
                (p_reg.index >= BASELINE[0])
                & (p_reg.index <= BASELINE[1])]
            if t_base_slice.empty or p_base_slice.empty:
                print(f'    {ssp} ({scheme}): baseline {BASELINE[0]}-'
                      f'{BASELINE[1]} not available')
                continue
            t_base = float(t_base_slice.mean())
            p_base = float(p_base_slice.mean())

            
            t_anom = t_reg - t_base
            p_anom = (p_reg - p_base) / p_base * 100.0

            results[scheme]['tas'][ssp][gcm] = t_anom
            results[scheme]['pr' ][ssp][gcm] = p_anom

        
        w = weights_area[valid_idx]
        w = w / w.sum()
        t_b = float((t_df * w).sum(axis=1).loc[
            BASELINE[0]:BASELINE[1]].mean())
        p_b = float((p_df * w).sum(axis=1).loc[
            BASELINE[0]:BASELINE[1]].mean())
        print(f'    {ssp:<7}  area-wt baseline T = {t_b:6.2f} degC   '
              f'P = {p_b:7.1f} mm yr-1   N = {len(valid_idx)}')

# ---------------------------------------------------------------------------
# Save per-scheme CSVs
# ---------------------------------------------------------------------------
print('\nSaving CSVs...')
for scheme in ('area', 'equal'):
    rows = []
    for var in ('tas', 'pr'):
        unit = 'degC' if var == 'tas' else 'percent'
        for ssp in SSPS:
            df = results[scheme][var][ssp]
            if df.empty:
                continue
            mean = df.mean(axis=1)
            std  = df.std(axis=1)
            for yr in mean.index:
                rows.append({
                    'category':  CATEGORY,
                    'scheme':    f'{scheme}_weighted',
                    'variable':  var,
                    'unit':      unit,
                    'ssp':       ssp,
                    'year':      int(yr),
                    'mean':      float(mean.loc[yr]),
                    'std':       float(std.loc[yr]),
                    'n_gcm':     int(df.loc[yr].count()),
                })
    if rows:
        path = os.path.join(
            OUTDIR,
            f'{CATEGORY}_climate_anomalies_{scheme}_weighted.csv')
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f'  {path}')

# ---------------------------------------------------------------------------
# Plot one figure per weighting scheme
# ---------------------------------------------------------------------------
def plot_scheme(scheme, scheme_label):
    """Plot 2 rows (T, P) x 1 column for the given weighting scheme."""
    fig, axes = plt.subplots(2, 1, figsize=(10, 9), sharex=True)
    ax_t, ax_p = axes
    ax_t.set_title(f'{label} - climate projection anomalies '
                   f'(N = {len(gdirs)}, {scheme_label})\n'
                   f'relative to {BASELINE[0]}-{BASELINE[1]}, '
                   f'bias-corrected, lapse-rate-adjusted',
                   fontsize=11, fontweight='bold')

    text_y_t = 0.02
    text_y_p = 0.02
    any_data = False
    for ssp in SSPS:
        c = SSP_COLOURS[ssp]
        df_t = results[scheme]['tas'][ssp]
        df_p = results[scheme]['pr' ][ssp]

        if not df_t.empty:
            any_data = True
            m = df_t.mean(axis=1); s = df_t.std(axis=1)
            m_plot = m.loc[(m.index >= PLOT_START)
                           & (m.index <= PROJ_END[1])]
            s_plot = s.loc[(s.index >= PLOT_START)
                           & (s.index <= PROJ_END[1])]
            ax_t.plot(m_plot.index, m_plot.values, color=c, lw=1.4,
                      label=SSP_LABELS[ssp])
            ax_t.fill_between(m_plot.index,
                              m_plot - s_plot, m_plot + s_plot,
                              color=c, alpha=0.15)
            end_m = m.loc[PROJ_END[0]:PROJ_END[1]].mean()
            end_s = s.loc[PROJ_END[0]:PROJ_END[1]].mean()
            ax_t.text(0.98, text_y_t,
                      f'{PROJ_END[0]}-{PROJ_END[1]}: '
                      f'{end_m:+.1f} \u00b1 {end_s:.1f} \u00b0C',
                      color=c, fontsize=9, ha='right',
                      transform=ax_t.transAxes)
            text_y_t += 0.05

        if not df_p.empty:
            m = df_p.mean(axis=1); s = df_p.std(axis=1)
            m_plot = m.loc[(m.index >= PLOT_START)
                           & (m.index <= PROJ_END[1])]
            s_plot = s.loc[(s.index >= PLOT_START)
                           & (s.index <= PROJ_END[1])]
            ax_p.plot(m_plot.index, m_plot.values, color=c, lw=1.4,
                      label=SSP_LABELS[ssp])
            ax_p.fill_between(m_plot.index,
                              m_plot - s_plot, m_plot + s_plot,
                              color=c, alpha=0.15)
            end_m = m.loc[PROJ_END[0]:PROJ_END[1]].mean()
            end_s = s.loc[PROJ_END[0]:PROJ_END[1]].mean()
            ax_p.text(0.98, text_y_p,
                      f'{PROJ_END[0]}-{PROJ_END[1]}: '
                      f'{end_m:+.0f} \u00b1 {end_s:.0f}%',
                      color=c, fontsize=9, ha='right',
                      transform=ax_p.transAxes)
            text_y_p += 0.05

    if not any_data:
        plt.close(fig)
        return None

    ax_t.axhline(0, color='grey', lw=0.5, ls='--')
    ax_p.axhline(0, color='grey', lw=0.5, ls='--')
    ax_t.set_xlim(PLOT_START, PROJ_END[1])
    ax_p.set_xlim(PLOT_START, PROJ_END[1])
    ax_t.grid(alpha=0.25)
    ax_p.grid(alpha=0.25)
    ax_t.set_ylabel('Temperature anomaly (\u00b0C)')
    ax_p.set_ylabel('Precipitation anomaly (%)')
    ax_p.set_xlabel('Year')
    ax_t.legend(fontsize=10, loc='upper left')

    fig.tight_layout()
    fig_path = os.path.join(
        OUTDIR,
        f'{CATEGORY}_climate_anomalies_{scheme}_weighted.png')
    fig.savefig(fig_path, dpi=200, bbox_inches='tight')
    plt.show()
    return fig_path

print('\nMaking figures...')
fp_area  = plot_scheme('area',  'area-weighted')
fp_equal = plot_scheme('equal', 'equal-weighted')
print(f'\n  area-weighted figure:  {fp_area}')
print(f'  equal-weighted figure: {fp_equal}')

# ---------------------------------------------------------------------------
# Summary block
# ---------------------------------------------------------------------------
print('\n' + '=' * 72)
print(f'SUMMARY - {label} climate anomalies')
print('=' * 72)
print(f'  Glaciers: {len(gdirs)}')
print(f'  Total area: {total_area:.1f} km^2')
print(f'  Baseline: {BASELINE[0]}-{BASELINE[1]}')
print(f'  GCM ensemble: {len(GCMS)} CMIP6 models')
print()
for scheme in ('area', 'equal'):
    print(f'  --- {scheme.upper()}-WEIGHTED ---')
    print(f'  {"SSP":<10}{"dT 2071-2100 (degC)":>22}'
          f'{"dP 2071-2100 (%)":>22}')
    for ssp in SSPS:
        df_t = results[scheme]['tas'][ssp]
        df_p = results[scheme]['pr' ][ssp]
        if df_t.empty or df_p.empty:
            print(f'  {SSP_LABELS[ssp]:<10}     -                     -')
            continue
        t_m = df_t.mean(axis=1).loc[PROJ_END[0]:PROJ_END[1]].mean()
        t_s = df_t.std(axis=1).loc[PROJ_END[0]:PROJ_END[1]].mean()
        p_m = df_p.mean(axis=1).loc[PROJ_END[0]:PROJ_END[1]].mean()
        p_s = df_p.std(axis=1).loc[PROJ_END[0]:PROJ_END[1]].mean()
        print(f'  {SSP_LABELS[ssp]:<10}'
              f'{t_m:>+12.1f} +/- {t_s:>4.1f}'
              f'{p_m:>+14.0f} +/- {p_s:>4.0f}')
    print()
print('=' * 72)
print('\nTo run next region: change CATEGORY at the top and re-run.')